In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Preprocessing data}}$

In [ ]:
df_temp = pd.read_csv(ROOT / "data/complete_dataframe2_30min.csv",delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv", parse_dates=["timestamp"])
df_price = pd.read_csv(ROOT / "data/DayAheadData_fulltimeperiod.csv")
df_price["ts"] = pd.to_datetime(df_price['ts'])
df_price = df_price.set_index("ts") # set time as index



filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14") &
    (df_load["timestamp"] <= "2022-04-11")
]
# How many 15-min readings are there available per meter_id?
counts = filtered.groupby("meter_id").size().reset_index(name="num_readings")

In [ ]:
df_temp2 = df_temp.join(df_price,how="inner")
df_temp2 = df_temp2[:-1]

In [ ]:
downsizing_factor = 0.01
total_load = filtered.groupby("timestamp")["consumption"].sum().reset_index(name="total_consumption")
total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])+pd.DateOffset(years=4)
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean() * downsizing_factor

In [ ]:
df = df_temp2.join(total_load_30min,how="inner")

In [ ]:
# Confirm data has 48-step days
total_load_30min["date"] = total_load_30min.index.date
load_counts = total_load_30min.groupby("date").size()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

# Make days into 48-step vectors

load_day_profiles = []

for _, day_df in load_days_df.groupby("date"):
    day_profile = day_df.sort_index()["total_consumption"].to_numpy()
    load_day_profiles.append(day_profile)

# Visualization of daily profiles <-- Would the MW scale be per time step or across the day profile?

daily_totals = total_load_30min.groupby(total_load_30min.index.date)["total_consumption"].sum()
daily_energy = daily_totals * 0.5

In [ ]:
# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)
df_agg = df_agg.drop(columns=["Aircon_WT Power_m", "Gaia_WT Power_m","B117_m", "B319_m", "B330_1_m", "B330_2_m",
                               "B330_3_m", "B716_m", "B715_m","wind_dir","wind_max","wind_min","wind_speed","radia_glob","temp_dry"])
# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

In [ ]:
df_agg

## $\text{\textcolor{green}{Generation vs Load}}$

In [ ]:
df_grouped = {str(date): group for date, group in df_agg.groupby("date")}

In [ ]:
## PLOT DAILY PROFILE OF LOAD AND POWER 
days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]
cols = ["PV_total","Wind_total","total_consumption"]
# Calculate number of subplots needed
n_days = len(days)
n_cols = 1
n_rows = 4

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 10))
axes = axes.flatten()  # Flatten to easily index

# Plot each day
for i, day in enumerate(days):
    df_day = df_grouped[day]  
    
    # Plot different parameters (columns) vs time
    # Assuming 'timestamp' or 'time' is your x-axis column
    for column in cols:
        axes[i].plot(df_day.index.strftime('%H:%M'), df_day[column], label=column, marker='o')
    
    axes[i].set_title(f'Day: {day}')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('kW')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## $\text{\textcolor{green}{Implementing the solver with "Perfect Knowledge" }}$

In [ ]:
# Daily solver
def solve_day_profile(df, minimum_self_sufficiency, E_capacity, P_capacity, charge_eff, discharge_eff, dT, C_rate):
    PV_gen = df["PV_total"].to_numpy()
    Wind_gen = df["Wind_total"].to_numpy()
    n = len(df)
    load = df["total_consumption"].to_numpy()
    price = df["DayAheadPriceDKK"].to_numpy()
    
    if P_capacity > C_rate * E_capacity:
        print(f"Warning: P_capacity={P_capacity}kW exceeds C_rate * E_capacity = {C_rate * E_capacity}")

    # Constants
    Pload = cp.Constant(load)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)
    P_capacity = cp.Constant(P_capacity)
    E_capacity = cp.Constant(E_capacity)

    # Variables

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)

    Pgrid_buy = cp.Variable(n) # if positive, energy is being bought from the grid. if negative we are selling energy to the grid. 

    SOC = cp.Variable(n)

    constraints = [
        Pcharge <= P_capacity, #### should it include the mask z or not? 
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity # Do we want this?  
        ]


    cost_energy_buy = cp.sum(cp.multiply(Pgrid_buy, price)) * dT 
    #DKK 
    

    problem = cp.Problem(cp.Minimize(cost_energy_buy), constraints)
    
    try:
        problem.solve(solver=cp.OSQP, verbose=False,max_iter=1000000)



        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"Problem arrose: {problem.status}")
            return {
            "status": problem.status,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
            }
        
        Pgrid_sell = cp.neg(Pgrid_buy)
        Pgrid_buy = cp.pos(Pgrid_buy)

        return {
        
            "status": problem.status,
            "Total_cost_DKK": (cost_energy_buy).value,
            "Cost_of_bought_energy_DKK": cp.sum(cp.multiply(Pgrid_buy, price)) * dT,
            "bought_energy_kWh": np.sum(Pgrid_buy.value) * dT,
            "sold_energy_kWh": np.sum(Pgrid_sell.value) * dT,

            "Pgrid_buy": Pgrid_buy.value,
            "Pgrid_sell": Pgrid_sell.value,
            "Pcharge": Pcharge.value,
            "Pdischarge": Pdischarge.value,
            "SOC": SOC.value
        }

    except Exception as e:
        print(f"Error arose {type(e).__name__}")
        return {
            "status": type(e).__name__,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
        }

## $\text{\textcolor{green}{Running Solver}}$

In [ ]:
# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 1
P_capacity = 10 ###### ADD RESULT FROM EMIs OPTIMIZATION 
E_capacity = 10 ####### ADD RESULT FROM EMIs OPTIMIZATION 

min_self_sufficiency = 0 # NO REQUIREMENT

In [ ]:
# Run the Optimization for the given days
profiles = []

days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]

for day in days:
    print("---------------------------------------------------------")
    print("Optimization starting for ", day)
    print("---------------------------------------------------------")
    day_df = df_grouped[day]

    out = solve_day_profile(
        day_df,
        min_self_sufficiency,
        E_capacity,
        P_capacity,
        charge_eff,
        discharge_eff,
        dT,
        C_rate
    )

    # Build time-series dataframe
    profile_df = pd.DataFrame({
        "ts": day_df.index,
        "P_buy": out["Pgrid_buy"],
        "P_sell": out["Pgrid_sell"],
        "P_charge": out["Pcharge"],
        "P_discharge": out["Pdischarge"],

        "SOC": out["SOC"],
        "price": day_df["DayAheadPriceDKK"].values,
        "load": day_df["total_consumption"].values,
        "PV": day_df["PV_total"].values,
        "Wind": day_df["Wind_total"].values,
    })

    profile_df["date"] = day
    print(f"Status for the optimization for {day}: {out["status"]} ")
    print(f"Total cost is: {out["Total_cost_DKK"]}")
    print(f"Amount of bought energy is: {out["bought_energy_kWh"]}")
    print(f"Amount of sold energy is: {out["sold_energy_kWh"]}")

    profiles.append(profile_df)


profiles_df = pd.concat(profiles, ignore_index=True)
profiles_df = profiles_df.set_index("ts")

## $\text{\textcolor{green}{Analyzing Solver Output}}$

In [ ]:
# Plot the Daily Power Profiles
import matplotlib.dates as mdates
days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"] # four random days spread across the year 

pos_cols = ["PV",	"Wind", "P_discharge", "P_buy"]
pos_color = ["gold","lightskyblue","plum","lightgreen"]
labels = ["PV", "Wind", "Battery", "Grid"]

neg_cols = ["P_charge","P_sell"]
neg_color = ["plum","lightgreen"]

n_days = len(days)
n_cols = 2
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 9))

axes = axes.flatten()  # Flatten to easily index


for j, day in enumerate(days):

    day_plot = profiles_df[profiles_df["date"] == day]


    width = 0.85/len(day_plot) # the width of the bars: can also be len(x) sequence


    
    top = np.zeros(len(day_plot))
    bottom = np.zeros(len(day_plot))
    for i,col in enumerate(pos_cols): # Plotting all "positive" power
        color = pos_color[i]
        axes[j].bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color = color)

        top += day_plot[col]

    for i,col in enumerate(neg_cols): # Plotting all "negative" power
        color = neg_color[i]
        axes[j].bar(day_plot.index, - day_plot[col], width, bottom=bottom, color=color)

        bottom -= day_plot[col]

    axes[j].scatter(day_plot.index,day_plot["load"],label="Load",marker="o",color="lightsalmon") # Plot Load
    axes[j].set_xlabel("Time (hour)")
    axes[j].set_ylabel("Power (kW)")
    axes[j].set_title(day)
    
    axes[j].xaxis.set_major_locator(mdates.HourLocator(interval=2))
    axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%H'))


handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles))
fig.tight_layout()
fig.show()

In [ ]:
test_days = ['2025-05-15',
 '2025-05-16',
 '2025-05-17',
 '2025-05-18',
 '2025-05-19',
 '2025-05-20',
 '2025-05-21',
 '2025-05-22',
 '2025-05-23',
 '2025-05-24',
 '2025-05-25',
 '2025-05-26',
 '2025-05-27',
 '2025-05-28',
 '2025-05-29',
 '2025-05-30',
 '2025-05-31',
 '2025-06-01',
 '2025-06-02',
 '2025-06-03',
 '2025-06-04',
 '2025-06-05',
 '2025-06-06',
 '2025-06-08',
 '2025-06-09',
 '2025-06-10',
 '2025-06-11',
 '2025-06-12',
 '2025-06-13',
 '2025-06-14',
 '2025-06-15',
 '2025-06-16',
 '2025-06-17',
 '2025-06-18',
 '2025-06-19',
 '2025-06-20',
 '2025-06-21',
 '2025-06-22',
 '2025-06-23',
 '2025-06-24',
 '2025-06-25',
 '2025-06-26',
 '2025-06-27',
 '2025-06-28',
 '2025-06-29',
 '2025-06-30',
 '2025-07-01',
 '2025-07-02',
 '2025-07-03',
 '2025-07-04',
 '2025-07-05',
 '2025-07-06',
 '2025-07-07',
 '2025-07-08',
 '2025-07-09',
 '2025-07-10',
 '2025-07-11',
 '2025-07-12',
 '2025-07-13',
 '2025-07-14',
 '2025-07-15',
 '2025-07-16',
 '2025-07-17',
 '2025-07-18',
 '2025-07-19',
 '2025-07-20',
 '2025-07-21',
 '2025-07-22',
 '2025-07-23',
 '2025-07-24',
 '2025-07-25',
 '2025-07-26',
 '2025-07-27',
 '2025-07-28',
 '2025-07-29',
 '2025-07-30',
 '2025-07-31',
 '2025-08-01',
 '2025-08-02',
 '2025-08-03',
 '2025-08-04',
 '2025-08-05',
 '2025-08-06',
 '2025-08-07',
 '2025-08-08',
 '2025-08-09',
 '2025-08-10',
 '2025-08-11',
 '2025-08-13',
 '2025-08-14',
 '2025-08-16',
 '2025-08-18',
 '2025-08-19',
 '2025-08-20',
 '2025-08-21',
 '2025-08-22',
 '2025-08-23',
 '2025-08-24',
 '2025-08-25',
 '2025-08-27',
 '2025-08-30',
 '2025-08-31',
 '2025-09-01',
 '2025-09-02',
 '2025-09-03',
 '2025-09-04',
 '2025-09-05',
 '2025-09-06',
 '2025-09-07',
 '2025-09-08',
 '2025-09-09',
 '2025-09-10',
 '2025-09-11',
 '2025-09-12',
 '2025-09-13',
 '2025-09-14',
 '2025-09-15',
 '2025-09-16',
 '2025-09-17',
 '2025-09-18',
 '2025-09-19',
 '2025-09-20',
 '2025-09-21',
 '2025-09-22',
 '2025-09-23',
 '2025-09-24',
 '2025-09-25',
 '2025-09-26',
 '2025-09-27',
 '2025-09-28',
 '2025-09-29',
 '2025-09-30',
 '2025-10-01',
 '2025-10-02',
 '2025-10-03',
 '2025-10-04',
 '2025-10-05',
 '2025-10-06',
 '2025-10-07',
 '2025-10-08',
 '2025-10-09',
 '2025-10-10',
 '2025-10-11',
 '2025-10-12',
 '2025-10-13',
 '2025-10-14',
 '2025-10-15',
 '2025-10-16',
 '2025-10-17',
 '2025-10-18',
 '2025-10-19',
 '2025-10-20',
 '2025-10-21',
 '2025-10-22',
 '2025-10-23',
 '2025-10-24',
 '2025-10-25',
 '2025-10-26',
 '2025-10-27',
 '2025-10-28',
 '2025-10-29',
 '2025-10-30',
 '2025-10-31',
 '2025-11-01',
 '2025-11-02',
 '2025-11-03',
 '2025-11-04',
 '2025-11-05',
 '2025-11-08',
 '2025-11-09',
 '2025-11-10',
 '2025-11-11',
 '2025-11-12',
 '2025-11-13',
 '2025-11-14',
 '2025-11-15',
 '2025-11-16',
 '2025-11-17',
 '2025-11-18',
 '2025-11-19',
 '2025-11-20',
 '2025-11-21',
 '2025-11-22',
 '2025-11-23',
 '2025-11-24',
 '2025-11-25',
 '2025-11-26',
 '2025-11-27',
 '2025-11-28',
 '2025-11-29',
 '2025-11-30',
 '2025-12-01',
 '2025-12-02',
 '2025-12-03',
 '2025-12-04',
 '2025-12-05',
 '2025-12-06',
 '2025-12-07',
 '2025-12-08',
 '2025-12-09',
 '2025-12-10',
 '2025-12-11',
 '2025-12-12',
 '2025-12-13',
 '2025-12-14',
 '2025-12-15',
 '2025-12-16',
 '2025-12-17',
 '2025-12-18',
 '2025-12-19',
 '2025-12-20',
 '2025-12-21',
 '2025-12-22',
 '2025-12-23',
 '2025-12-24',
 '2025-12-25',
 '2025-12-26',
 '2025-12-27',
 '2025-12-28',
 '2025-12-29',
 '2025-12-30',
 '2025-12-31',
 '2026-01-02',
 '2026-01-03',
 '2026-01-04',
 '2026-01-05',
 '2026-01-06',
 '2026-01-07',
 '2026-01-08',
 '2026-01-09',
 '2026-01-10',
 '2026-01-11',
 '2026-01-12',
 '2026-01-13',
 '2026-01-14',
 '2026-01-15',
 '2026-01-16',
 '2026-01-17',
 '2026-01-19',
 '2026-01-20',
 '2026-01-21',
 '2026-01-22',
 '2026-01-23',
 '2026-01-24',
 '2026-01-25',
 '2026-01-26',
 '2026-01-27',
 '2026-01-28',
 '2026-01-29',
 '2026-01-30',
 '2026-01-31',
 '2026-02-01',
 '2026-02-02',
 '2026-02-03',
 '2026-02-04',
 '2026-02-05',
 '2026-02-06',
 '2026-02-07',
 '2026-02-08',
 '2026-02-09',
 '2026-02-10',
 '2026-02-11',
 '2026-02-12',
 '2026-02-13',
 '2026-02-14',
 '2026-02-15',
 '2026-02-16',
 '2026-02-17',
 '2026-02-18',
 '2026-02-19',
 '2026-02-20',
 '2026-02-21',
 '2026-02-22',
 '2026-02-23',
 '2026-02-24',
 '2026-02-25',
 '2026-02-26',
 '2026-02-27',
 '2026-02-28',
 '2026-03-01',
 '2026-03-02',
 '2026-03-03',
 '2026-03-04',
 '2026-03-05',
 '2026-03-06',
 '2026-03-07',
 '2026-03-08',
 '2026-03-09',
 '2026-03-10',
 '2026-03-11',
 '2026-03-12',
 '2026-03-13',
 '2026-03-14',
 '2026-03-15',
 '2026-03-16',
 '2026-03-17',
 '2026-03-21',
 '2026-03-22',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28',
 '2026-03-29',
 '2026-03-30',
 '2026-03-31',
 '2026-04-01',
 '2026-04-02',
 '2026-04-03',
 '2026-04-04',
 '2026-04-05',
 '2026-04-06',
 '2026-04-07',
 '2026-04-08',
 '2026-04-09']